In [ ]:
# Method 2 - EfficientNet-B0 + BiLSTM + Bilinear Fusion  [PathVQA]
# Changes 1,2,4,6 + Change 8: TRUE Grad-CAM on EfficientNet conv features
# (this model has no attention, so Grad-CAM is the correct saliency method).

# =========================
# 0. INSTALLS
# =========================
!pip -q install datasets timm nltk rouge-score scikit-learn

# =========================
# 1. IMPORTS
# =========================
import os
import string
import re
import gc
import random
import hashlib
import warnings
from io import BytesIO
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

from datasets import load_dataset
import timm
from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

warnings.filterwarnings("ignore")

# =========================
# 2. SETTINGS
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEVICE: {DEVICE}")

# Config
IMAGE_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 15
LR = 1e-4
WEIGHT_DECAY = 1e-4
MAX_Q_LEN = 30
WORD_VOCAB_SIZE = 2200
EMBED_DIM = 300
LSTM_HIDDEN = 512
LSTM_DROPOUT = 0.4

OUTPUT_DIR = "/kaggle/working/pathvqa_all_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

STOP_WORDS = set(stopwords.words("english"))
SMOOTH_FN = SmoothingFunction().method1
ROUGE_SCORER = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

FUSION_METHODS = ["bilinear"]

# =========================
# 3. TEXT HELPERS
# =========================
# -------------------------------------------------------------------------
# CHANGE 2: Fixed normalization.
# We NO LONGER strip stopwords. Stripping stopwords was corrupting answers:
# "no" is an NLTK stopword -> every "no" answer became <unk_a>, and
# "not present" -> "present" (negation inverted). Answers are now normalized
# with lowercase + punctuation strip + whitespace collapse ONLY. Questions
# get the same light normalization (medical questions are short; removing
# stopwords loses meaning).
# -------------------------------------------------------------------------
def _basic_clean(x):
    x = str(x).lower().strip()
    x = x.translate(str.maketrans("", "", string.punctuation))
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def normalize_question(x, unk_token="<unk_q>"):
    x = _basic_clean(x)
    return x if x else unk_token

def normalize_answer(x, unk_token="<unk_a>"):
    x = _basic_clean(x)
    return x if x else unk_token

# CHANGE 6 helper: derive the question type (What / Where / How / Yes-No /
# other) from the raw question and raw answer, so per-type reporting does not
# depend on an optional dataset column. Yes/No is decided by the answer
# (most reliable); open-ended types by the leading question word.
_YESNO_LEAD = {"is", "are", "does", "do", "did", "was", "were", "can",
               "could", "has", "have", "will", "should", "would", "may", "am"}
def get_question_type(question_raw, answer_raw):
    a = str(answer_raw).strip().lower()
    if a in ("yes", "no"):
        return "yes/no"
    q = _basic_clean(question_raw)
    first = q.split()[0] if q.split() else ""
    if first in ("what", "which", "whose", "name"):
        return "what"
    if first == "where":
        return "where"
    if first == "how":
        return "how"
    if first == "when":
        return "when"
    if first in _YESNO_LEAD:
        return "yes/no"
    return "other"

def tokenize_text(x):
    toks = str(x).strip().split()
    return toks if toks else ["<unk>"]

def encode_question(text, stoi, max_len=MAX_Q_LEN, unk_idx=1, pad_idx=0):
    toks = tokenize_text(text)[:max_len]
    ids = [stoi.get(tok, unk_idx) for tok in toks]
    q_len = max(1, len(ids))
    ids += [pad_idx] * (max_len - len(ids))
    return ids, q_len

def image_to_pil(img_obj):
    if isinstance(img_obj, Image.Image):
        return img_obj.convert("RGB")
    if isinstance(img_obj, dict):
        if img_obj.get("bytes") is not None:
            return Image.open(BytesIO(img_obj["bytes"])).convert("RGB")
        if img_obj.get("path") is not None:
            return Image.open(img_obj["path"]).convert("RGB")
    return Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE))

def stable_image_id(img_obj, fallback="unknown"):
    try:
        if isinstance(img_obj, dict):
            if img_obj.get("path") is not None:
                return str(img_obj["path"])
            if img_obj.get("bytes") is not None:
                return hashlib.md5(img_obj["bytes"]).hexdigest()
        if isinstance(img_obj, Image.Image):
            img_rgb = img_obj.convert("RGB")
            buf = BytesIO()
            img_rgb.save(buf, format="PNG")
            return hashlib.md5(buf.getvalue()).hexdigest()
    except Exception:
        return str(fallback)
    return str(fallback)

# =========================
# 4. LOAD DATASET
# =========================
print("\nLOADING DATASET...")
dataset = load_dataset("flaviagiammarino/path-vqa")

base_cols = dataset["train"].column_names
IMAGE_COL = "image" if "image" in base_cols else "img"
QUESTION_COL = "question" if "question" in base_cols else "query"
ANSWER_COL = "answer" if "answer" in base_cols else "answers"
CATEGORY_COL = "answer_type" if "answer_type" in base_cols else None
IMAGE_ID_COL = "image_id" if "image_id" in base_cols else None

print(f"Using: image={IMAGE_COL}, question={QUESTION_COL}, answer={ANSWER_COL}")

# =========================
# 5. BUILD DATAFRAMES FROM THE OFFICIAL SPLITS
# =========================
# CHANGE 1: Use the dataset's OWN train / validation / test splits directly.
# The previous version merged all three splits into one dataframe and then
# re-shuffled the images into a fresh random 50/30/20 partition (seeded by 42).
# That matched the official counts (2499/1499/1000) but NOT the official
# images, so results were not comparable to He et al., BaMCo, or anyone else,
# and the paper's "standard split" claim was false. We now build one dataframe
# per official split and never re-partition.
def build_split_df(hf_split, split_name):
    rows = []
    for i in range(len(hf_split)):
        ex = hf_split[i]
        img_obj = ex[IMAGE_COL]
        image_id = str(ex[IMAGE_ID_COL]) if (IMAGE_ID_COL is not None and IMAGE_ID_COL in ex) else stable_image_id(img_obj, fallback=f"{split_name}_{i}")

        q_raw = str(ex[QUESTION_COL])
        a_raw = str(ex[ANSWER_COL])

        q_proc = normalize_question(q_raw)   # CHANGE 2
        a_proc = normalize_answer(a_raw)     # CHANGE 2

        rows.append({
            "source_split": split_name,
            "source_idx": i,
            "image_id": image_id,
            "question_raw": q_raw,
            "answer_raw": a_raw,
            "question_proc": q_proc,
            "answer_proc": a_proc,
            # CHANGE 6: robust question-type label for per-type reporting
            "question_type": get_question_type(q_raw, a_raw),
        })
    return pd.DataFrame(rows)

train_df = build_split_df(dataset["train"], "train").reset_index(drop=True)
val_df   = build_split_df(dataset["validation"], "validation").reset_index(drop=True)
test_df  = build_split_df(dataset["test"], "test").reset_index(drop=True)

print("\n===== OFFICIAL PATHVQA SPLIT =====")
print(f"Train images: {train_df['image_id'].nunique()} | Train QA: {len(train_df)}")
print(f"Val images:   {val_df['image_id'].nunique()} | Val QA:   {len(val_df)}")
print(f"Test images:  {test_df['image_id'].nunique()} | Test QA:  {len(test_df)}")
print("\nTest question-type distribution:")
print(test_df["question_type"].value_counts())

# =========================
# 7. WORD VOCAB
# =========================
word_counter = Counter()
for text in train_df["question_proc"].tolist() + train_df["answer_proc"].tolist():
    word_counter.update(tokenize_text(text))

SPECIAL_TOKENS = ["<pad>", "<unk>"]
most_common_words = [w for w, _ in word_counter.most_common(WORD_VOCAB_SIZE - len(SPECIAL_TOKENS))]
itos = SPECIAL_TOKENS + most_common_words
stoi = {w: i for i, w in enumerate(itos)}
UNK_IDX = stoi["<unk>"]
PAD_IDX = stoi["<pad>"]

print(f"Word vocab size: {len(stoi)}")

def encode_question_text(text, max_len=MAX_Q_LEN):
    toks = tokenize_text(text)[:max_len]
    ids = [stoi.get(tok, UNK_IDX) for tok in toks]
    q_len = max(1, len(ids))
    ids += [PAD_IDX] * (max_len - len(ids))
    return ids, q_len

# =========================
# 8. ANSWER SPACE
# =========================
train_answer_counts = Counter(train_df["answer_proc"].tolist())
answer_list = sorted(train_answer_counts.keys())
if "<unk_a>" not in answer_list:
    answer_list = answer_list + ["<unk_a>"]

answer2id = {a: i for i, a in enumerate(answer_list)}
id2answer = {i: a for a, i in answer2id.items()}
print(f"Answer classes: {len(answer2id)}")

def map_answer_to_id(answer_text):
    return answer2id.get(answer_text, answer2id["<unk_a>"])

# =========================
# 9. TRANSFORMS
# =========================
class RandomAffinePaper:
    def __call__(self, img):
        angle = random.uniform(-5, 5)
        tx = int(random.uniform(-0.08, 0.08) * img.size[0])
        ty = int(random.uniform(-0.08, 0.08) * img.size[1])
        scale = random.uniform(0.92, 1.08)
        shear = random.uniform(-8, 8)
        return TF.affine(img, angle=angle, translate=(tx, ty), scale=scale, shear=shear)

train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    RandomAffinePaper(),
    T.ToTensor()
])

eval_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor()
])

# =========================
# 10. SIMPLE MODEL
# =========================
class SimpleVQAModel(nn.Module):
    def __init__(self, fusion_type, vocab_size, num_answers):
        super().__init__()
        self.fusion_type = fusion_type
        
        self.image_encoder = timm.create_model("efficientnet_b0", pretrained=True, features_only=True, out_indices=[-1])
        img_dim = self.image_encoder.feature_info.channels()[-1]
        
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(EMBED_DIM, LSTM_HIDDEN, batch_first=True, bidirectional=True)
        self.text_proj = nn.Linear(LSTM_HIDDEN * 2, LSTM_HIDDEN)
        self.dropout = nn.Dropout(LSTM_DROPOUT)
        if fusion_type == "bilinear":
            self.fusion = nn.Bilinear(img_dim, LSTM_HIDDEN, LSTM_HIDDEN)
        else:
            self.fusion = None
        
        self.classifier = nn.Sequential(
            nn.Linear(LSTM_HIDDEN, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_answers)
        )
        
    def forward(self, images, q_ids, q_lens):
        img_feat = self.image_encoder(images)[0]
        img_feat = F.adaptive_avg_pool2d(img_feat, 1).squeeze(-1).squeeze(-1)
        
        x = self.embedding(q_ids)
        packed = nn.utils.rnn.pack_padded_sequence(x, q_lens.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        txt_feat = self.text_proj(torch.cat([h[-2], h[-1]], dim=1))
        txt_feat = self.dropout(txt_feat)
        
        
        if self.fusion_type == "bilinear":
            fused = self.fusion(img_feat, txt_feat)
        return self.classifier(fused)

# =========================
# 11. DATASET
# =========================
class PathVQADataset(Dataset):
    def __init__(self, dataset_dict, df, transform):
        self.dataset_dict = dataset_dict
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        split_name = row["source_split"]
        source_idx = int(row["source_idx"])
        ex = self.dataset_dict[split_name][source_idx]

        img = image_to_pil(ex[IMAGE_COL])
        img = self.transform(img)

        q_proc = str(row["question_proc"]).strip()
        if q_proc == "":
            q_proc = "<unk_q>"

        a_proc = str(row["answer_proc"]).strip()
        if a_proc == "":
            a_proc = "<unk_a>"

        q_ids, q_len = encode_question_text(q_proc, MAX_Q_LEN)
        label = map_answer_to_id(a_proc)

        return {
            "image": img,
            "question_ids": torch.tensor(q_ids, dtype=torch.long),
            "question_len": torch.tensor(q_len, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.long),
            "question_raw": str(row["question_raw"]),
            "question_proc": q_proc,
            "answer_proc": a_proc,
            "image_id": str(row["image_id"]),
            "question_type": str(row["question_type"])  # CHANGE 6
        }

# =========================
# 12. METRICS
# =========================
def answer_token_f1(pred, gold):
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    
    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0
    
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def bleu_n(pred, gold, n):
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    weights = {1: (1.0, 0.0, 0.0, 0.0), 2: (0.5, 0.5, 0.0, 0.0), 3: (1/3, 1/3, 1/3, 0.0)}[n]
    try:
        return sentence_bleu([gold_tokens], pred_tokens, weights=weights, smoothing_function=SMOOTH_FN)
    except:
        return 0.0

def rouge_scores(pred, gold):
    try:
        scores = ROUGE_SCORER.score(gold, pred)
        return scores["rouge1"].fmeasure, scores["rougeL"].fmeasure
    except:
        return 0.0, 0.0

def compute_metrics_from_texts(golds, preds):
    yesno_idx = [i for i, g in enumerate(golds) if g in ["yes", "no"]]
    open_idx = [i for i, g in enumerate(golds) if g not in ["yes", "no"]]
    
    metrics = {}
    overall_exact = np.mean([int(g == p) for g, p in zip(golds, preds)]) * 100.0
    metrics["overall_exact_match_pct"] = float(overall_exact)
    
    if len(yesno_idx) > 0:
        y_true = [golds[i] for i in yesno_idx]
        y_pred = [preds[i] for i in yesno_idx]
        metrics["yes_no_accuracy_pct"] = float(accuracy_score(y_true, y_pred) * 100.0)
    else:
        metrics["yes_no_accuracy_pct"] = np.nan
    
    if len(open_idx) > 0:
        o_true = [golds[i] for i in open_idx]
        o_pred = [preds[i] for i in open_idx]
        
        exacts = [int(g == p) for g, p in zip(o_true, o_pred)]
        f1s = [answer_token_f1(p, g) for g, p in zip(o_true, o_pred)]
        b1 = [bleu_n(p, g, 1) for g, p in zip(o_true, o_pred)]
        b2 = [bleu_n(p, g, 2) for g, p in zip(o_true, o_pred)]
        b3 = [bleu_n(p, g, 3) for g, p in zip(o_true, o_pred)]
        r1, rl = [], []
        for g, p in zip(o_true, o_pred):
            s1, sL = rouge_scores(p, g)
            r1.append(s1)
            rl.append(sL)
        
        metrics["open_exact_match_pct"] = float(np.mean(exacts) * 100.0)
        metrics["macro_f1_pct"] = float(np.mean(f1s) * 100.0)
        metrics["bleu1_pct"] = float(np.mean(b1) * 100.0)
        metrics["bleu2_pct"] = float(np.mean(b2) * 100.0)
        metrics["bleu3_pct"] = float(np.mean(b3) * 100.0)
        metrics["rouge1_f_pct"] = float(np.mean(r1) * 100.0)
        metrics["rougeL_f_pct"] = float(np.mean(rl) * 100.0)
    else:
        metrics["open_exact_match_pct"] = np.nan
        metrics["macro_f1_pct"] = np.nan
        metrics["bleu1_pct"] = np.nan
        metrics["bleu2_pct"] = np.nan
        metrics["bleu3_pct"] = np.nan
        metrics["rouge1_f_pct"] = np.nan
        metrics["rougeL_f_pct"] = np.nan
    
    metrics["total_samples"] = int(len(golds))
    return metrics

# CHANGE 6: accuracy broken down by question type (What / Where / How /
# Yes-No / other). Returns an ordered dict: {type: (exact_match_pct, n)}.
def per_type_accuracy(golds, preds, types):
    order = ["yes/no", "what", "where", "how", "when", "other"]
    buckets = {}
    for g, p, t in zip(golds, preds, types):
        buckets.setdefault(t, []).append(int(g == p))
    result = {}
    for t in order:
        if t in buckets and len(buckets[t]) > 0:
            hits = buckets[t]
            result[t] = (float(np.mean(hits) * 100.0), len(hits))
    # include any unexpected types not in `order`
    for t, hits in buckets.items():
        if t not in result:
            result[t] = (float(np.mean(hits) * 100.0), len(hits))
    return result

# =========================
# 13. TRAINING FUNCTIONS
# =========================
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc="Train", leave=False):
        images = batch["image"].to(DEVICE)
        q_ids = batch["question_ids"].to(DEVICE)
        q_lens = batch["question_len"].to(DEVICE)
        labels = batch["label"].to(DEVICE)
        
        logits = model(images, q_ids, q_lens)
        loss = F.cross_entropy(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(images)
    
    return total_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    all_golds = []
    all_preds = []
    all_types = []   # CHANGE 6

    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            images = batch["image"].to(DEVICE)
            q_ids = batch["question_ids"].to(DEVICE)
            q_lens = batch["question_len"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            
            logits = model(images, q_ids, q_lens)
            loss = F.cross_entropy(logits, labels)
            total_loss += loss.item() * len(images)
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            golds = labels.cpu().numpy()
            
            all_preds.extend([id2answer[p] for p in preds])
            all_golds.extend([id2answer[g] for g in golds])
            all_types.extend(list(batch["question_type"]))  # CHANGE 6

    metrics = compute_metrics_from_texts(all_golds, all_preds)
    metrics["loss"] = total_loss / len(loader.dataset)
    # CHANGE 6: per-question-type accuracy, plus raw lists for reporting
    metrics["per_type"] = per_type_accuracy(all_golds, all_preds, all_types)
    metrics["_golds"] = all_golds
    metrics["_preds"] = all_preds
    metrics["_types"] = all_types
    return metrics

# =========================
# 14. PRINT PREDICTIONS FUNCTION
# =========================
@torch.no_grad()
# =========================
# CHANGE 8: GRAD-CAM for Method 2
# Method 2 uses global bilinear fusion and has NO attention map, so we use
# true gradient-based Grad-CAM on the EfficientNet-B0 last conv feature map.
# The forward below mirrors SimpleVQAModel.forward exactly, but keeps the
# conv feature map in the graph so we can read its gradient.
# =========================
def gradcam_for_sample(model, image, q_ids, q_len, class_idx=None):
    model.eval()
    for pm in model.parameters():
        pm.requires_grad_(True)  # ensure grad flows to conv activations

    feat = model.image_encoder(image)[0]        # (1, C, h, w)
    feat.retain_grad()
    pooled = F.adaptive_avg_pool2d(feat, 1).squeeze(-1).squeeze(-1)

    x = model.embedding(q_ids)
    packed = nn.utils.rnn.pack_padded_sequence(x, q_len.cpu(), batch_first=True, enforce_sorted=False)
    _, (h, _) = model.lstm(packed)
    txt = model.text_proj(torch.cat([h[-2], h[-1]], dim=1))
    txt = model.dropout(txt)
    fused = model.fusion(pooled, txt)
    logits = model.classifier(fused)

    if class_idx is None:
        class_idx = int(logits.argmax(dim=1).item())
    model.zero_grad(set_to_none=True)
    logits[0, class_idx].backward()

    grads = feat.grad[0]                    # (C, h, w)
    weights = grads.mean(dim=(1, 2))        # (C,)
    cam = torch.relu((weights[:, None, None] * feat[0]).sum(0))  # (h, w)
    cam = cam - cam.min()
    if cam.max() > 0:
        cam = cam / cam.max()
    return cam.detach().cpu().numpy(), class_idx, logits.detach()

@torch.enable_grad()
def plot_gradcam_10(model, dataset_obj, model_name, num_samples=10):
    indices = list(range(len(dataset_obj)))
    random.Random(SEED).shuffle(indices)
    indices = indices[:min(num_samples, len(indices))]
    for k, idx in enumerate(indices, 1):
        sample = dataset_obj[idx]
        image = sample["image"].unsqueeze(0).to(DEVICE)
        q_ids = sample["question_ids"].unsqueeze(0).to(DEVICE)
        q_len = sample["question_len"].unsqueeze(0).to(DEVICE)

        cam, class_idx, logits = gradcam_for_sample(model, image, q_ids, q_len)
        pred = id2answer.get(class_idx, "<missing>")
        gold = str(sample["answer_proc"]).strip() or "<blank>"

        img_np = sample["image"].permute(1, 2, 0).cpu().numpy()
        mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
        img_np = np.clip(std * img_np + mean, 0, 1)

        print("=" * 100)
        print(f"Sample {k} | Q-type: {sample['question_type']}")
        print("Question :", sample["question_raw"])
        print("GT       :", repr(gold), "| Pred:", repr(pred),
              "|", "CORRECT" if gold == pred else "WRONG")

        plt.figure(figsize=(11, 5))
        plt.subplot(1, 2, 1); plt.imshow(img_np); plt.axis("off"); plt.title("Image")
        plt.subplot(1, 2, 2); plt.imshow(img_np)
        plt.imshow(cam, cmap="jet", alpha=0.5,
                   extent=(0, img_np.shape[1], img_np.shape[0], 0))
        plt.axis("off"); plt.title("Grad-CAM Overlay")
        plt.suptitle(f"{model_name}\nQ: {sample['question_raw']}\nGT: {gold} | Pred: {pred}", fontsize=10)
        plt.tight_layout(); plt.show()

def print_10_predictions(model, test_dataset, model_name, num_samples=10):
    """Print 10 random test predictions with images"""
    model.eval()
    
    # Get random indices
    indices = list(range(len(test_dataset)))
    random.shuffle(indices)
    indices = indices[:num_samples]
    
    print("\n" + "="*100)
    print(f"📊 10 TEST PREDICTIONS FOR {model_name.upper()} MODEL")
    print("="*100)
    
    correct_count = 0
    
    for i, idx in enumerate(indices, 1):
        sample = test_dataset[idx]
        
        # Get image and text
        image = sample["image"].unsqueeze(0).to(DEVICE)
        q_ids = sample["question_ids"].unsqueeze(0).to(DEVICE)
        q_len = sample["question_len"].unsqueeze(0).to(DEVICE)
        
        # Predict
        logits = model(image, q_ids, q_len)
        pred_idx = torch.argmax(logits, dim=1).item()
        pred_answer = id2answer.get(pred_idx, "<missing>")
        
        gt_answer = sample["answer_proc"]
        question = sample["question_raw"]
        is_correct = (pred_answer.lower().strip() == gt_answer.lower().strip())
        
        if is_correct:
            correct_count += 1
        
        # Print
        print(f"\n{'─'*80}")
        print(f"🔹 SAMPLE #{i}")
        print(f"📝 QUESTION: {question}")
        print(f"✅ GROUND TRUTH: {gt_answer}")
        print(f"🤖 PREDICTION: {pred_answer}")
        print(f"📊 CORRECT: {'✓ YES' if is_correct else '✗ NO'}")
        
        # Display image
        img_np = sample["image"].permute(1, 2, 0).cpu().numpy()
        img_np = np.clip(img_np, 0, 1)
        
        plt.figure(figsize=(8, 6))
        plt.imshow(img_np)
        plt.axis("off")
        plt.title(f"Sample #{i}\nQ: {question[:60]}\nGT: {gt_answer}\nPred: {pred_answer}", 
                 fontsize=9, color='green' if is_correct else 'red')
        plt.tight_layout()
        plt.show()
    
    print(f"\n{'='*80}")
    print(f"📈 {model_name.upper()} - Accuracy on {num_samples} samples: {correct_count}/{num_samples} ({correct_count/num_samples*100:.1f}%)")
    print("="*80)

# =========================
# 15. CREATE DATASETS
# =========================
train_ds = PathVQADataset(dataset, train_df, train_transform)
val_ds = PathVQADataset(dataset, val_df, eval_transform)
test_ds = PathVQADataset(dataset, test_df, eval_transform)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"\n✅ Train samples: {len(train_ds)}")
print(f"✅ Val samples: {len(val_ds)}")
print(f"✅ Test samples: {len(test_ds)}")

# =========================
# 16. TRAIN EACH MODEL AND PRINT PREDICTIONS
# =========================
all_results = []

for fusion in FUSION_METHODS:
    print("\n" + "="*80)
    print(f"🚀 TRAINING MODEL: {fusion.upper()}")
    print("="*80)
    
    # Create model
    model = SimpleVQAModel(fusion, len(stoi), len(answer2id)).to(DEVICE)
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    
    best_em = -1
    best_path = os.path.join(OUTPUT_DIR, f"{fusion}_best.pt")
    
    # Training loop
    for epoch in range(1, EPOCHS + 1):
        print(f"\n📚 Epoch {epoch}/{EPOCHS}")
        
        train_loss = train_epoch(model, train_loader, optimizer)
        val_metrics = evaluate(model, val_loader)
        
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Overall EM: {val_metrics['overall_exact_match_pct']:.2f}%")
        print(f"Val Yes/No: {val_metrics['yes_no_accuracy_pct']:.2f}%")
        
        if val_metrics["overall_exact_match_pct"] > best_em:
            best_em = val_metrics["overall_exact_match_pct"]
            torch.save(model.state_dict(), best_path)
            print(f"✓ Best model saved (EM: {best_em:.2f}%)")
    
    # Load best model and test
    model.load_state_dict(torch.load(best_path))
    test_metrics = evaluate(model, test_loader)
    
    print("\n" + "="*60)
    print(f"🎯 TEST RESULTS - {fusion.upper()}")
    print("="*60)
    _skip = {"per_type", "_golds", "_preds", "_types"}  # not scalar
    for k, v in test_metrics.items():
        if k in _skip:
            continue
        if isinstance(v, float):
            print(f"{k}: {v:.2f}")
        else:
            print(f"{k}: {v}")

    # ---------- CHANGE 6: per-question-type accuracy table ----------
    print("\n" + "-"*60)
    print(f"📋 PER-QUESTION-TYPE EXACT MATCH - {fusion.upper()}")
    print("-"*60)
    print(f"{'Question Type':<14}{'Exact Match (%)':>18}{'N':>10}")
    for qtype, (acc, n) in test_metrics["per_type"].items():
        print(f"{qtype:<14}{acc:>18.2f}{n:>10}")

    # store a flat, CSV-friendly row (drop the bulky list/dict fields)
    flat = {k: v for k, v in test_metrics.items() if k not in _skip}
    flat["Fusion"] = fusion
    for qtype, (acc, n) in test_metrics["per_type"].items():
        flat[f"acc_{qtype.replace('/', '_')}"] = acc
    all_results.append(flat)
    
    # ==================== PRINT 10 SAMPLE PREDICTIONS ====================
    print_10_predictions(model, test_ds, fusion, num_samples=10)
    # CHANGE 8: Grad-CAM overlays for the fusion model
    plot_gradcam_10(model, test_ds, f"EfficientNet-B0 + BiLSTM + Bilinear ({fusion})", num_samples=10)
    
    # Cleanup
    del model
    torch.cuda.empty_cache()
    gc.collect()

# =========================
# 17. FINAL RESULTS SUMMARY
# =========================
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values("overall_exact_match_pct", ascending=False)

print("\n" + "="*80)
print("🏆 FINAL RESULTS - ALL MODELS")
print("="*80)
print(results_df.to_string(index=False))

# Save results
results_df.to_csv(os.path.join(OUTPUT_DIR, "all_results.csv"), index=False)

# Plot comparison
plt.figure(figsize=(10, 6))
bars = plt.bar(results_df["Fusion"], results_df["overall_exact_match_pct"], 
               color=['#2E86AB', '#A23B72', '#F18F01'])
plt.ylabel("Overall Exact Match (%)")
plt.title("Model Comparison")
plt.ylim(0, 100)
for bar, val in zip(bars, results_df["overall_exact_match_pct"]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', 
             ha='center', fontweight='bold')
plt.savefig(os.path.join(OUTPUT_DIR, "model_comparison.png"), dpi=150)
plt.show()

print("\n" + "="*80)
print(f"✅ COMPLETE! All models saved to {OUTPUT_DIR}")
print("="*80)
